# MLP Hyperparameter Tuning on `micro_mobility_training_data_2025.csv`

Time-aware tuning for MLP using a chronological validation window (no random CV).


## Colab Setup

- Auto-detect Colab and mount Google Drive.
- Update `COLAB_PROJECT_ROOT` if your Drive path is different.


## Step 1: Environment and Libraries

Loads required libraries for preprocessing, tuning, modeling, and evaluation.


In [ ]:
from pathlib import Path
import json
import time
import itertools
import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterSampler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## Step 2: Paths and Runtime Config

Defines dataset/artifact paths and tuning budgets (`N_TRIALS`, `VAL_DAYS`, `HOLDOUT_DAYS`).


In [ ]:
# Paths and config
COLAB_PROJECT_ROOT = Path('/content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction')

IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = COLAB_PROJECT_ROOT
else:
    PROJECT_ROOT = Path.cwd().resolve().parents[1]

DATA_PATH = PROJECT_ROOT / 'data' / 'proceed' / 'micro_mobility_training_data_2025.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'model_training' / 'mlp' / 'tuned_v1'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_DAYS = 7
VAL_DAYS = 14
RANDOM_STATE = 42
N_TRIALS = 12

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_PATH:', DATA_PATH)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

NOTEBOOK_T0 = time.perf_counter()


## Step 3: Data Load and Chronological Split

Builds train/validation/test splits by date (no random split) to avoid leakage.


In [ ]:
data_prep_t0 = time.perf_counter()
# Load data and build split
cols = [
    'station', 'date', 'hour', 'net_demand',
    'lat', 'lng', 'hour_sin', 'hour_cos', 'day_of_week', 'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h'
]

df = pd.read_csv(DATA_PATH, usecols=cols)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'station', 'net_demand']).copy()

stations_sorted = sorted(df['station'].dropna().unique().tolist())
station_to_id = {s: i for i, s in enumerate(stations_sorted)}
df['station_id'] = df['station'].map(station_to_id).astype('int32')

feature_cols = [
    'station_id', 'hour', 'lat', 'lng', 'hour_sin', 'hour_cos',
    'day_of_week', 'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h'
]

max_date = df['date'].max().normalize()
test_cutoff = max_date - pd.Timedelta(days=HOLDOUT_DAYS)
val_cutoff = test_cutoff - pd.Timedelta(days=VAL_DAYS)

train_mask = df['date'] <= val_cutoff
val_mask = (df['date'] > val_cutoff) & (df['date'] <= test_cutoff)
test_mask = df['date'] > test_cutoff

X_train = df.loc[train_mask, feature_cols].astype('float32')
y_train = df.loc[train_mask, 'net_demand'].astype('float32')
X_val = df.loc[val_mask, feature_cols].astype('float32')
y_val = df.loc[val_mask, 'net_demand'].astype('float32')
X_test = df.loc[test_mask, feature_cols].astype('float32')
y_test = df.loc[test_mask, 'net_demand'].astype('float32')

print('Rows total:', len(df))
print('Train rows:', len(X_train))
print('Val rows:', len(X_val))
print('Test rows:', len(X_test))
print('Val cutoff:', val_cutoff.date())
print('Test cutoff:', test_cutoff.date())

data_prep_seconds = float(time.perf_counter() - data_prep_t0)
print(f'Data prep time: {data_prep_seconds:.2f} sec ({data_prep_seconds/60:.2f} min)')


## Step 4: Hyperparameter Search Space

Defines candidate values for MLP architecture and optimization settings.


In [ ]:
# Randomized search space
param_dist = {
    'hidden_layer_sizes': [(128,64), (256,128), (256,128,64), (128,128)],
    'alpha': [1e-5, 1e-4, 5e-4, 1e-3],
    'learning_rate_init': [5e-4, 1e-3, 2e-3],
    'batch_size': [1024, 2048, 4096],
    'max_iter': [35, 50],
}

trials = list(ParameterSampler(param_dist, n_iter=N_TRIALS, random_state=RANDOM_STATE))
print('Trials:', len(trials))


## Step 5: Tuning Loop

Runs randomized trials and tracks per-trial validation metrics + runtime.


In [ ]:
# Tune on train/val
trial_rows = []
best = None
tuning_t0 = time.perf_counter()

for idx, p in enumerate(trials, start=1):
    t0 = time.perf_counter()
    base_mlp = MLPRegressor(
        hidden_layer_sizes=p['hidden_layer_sizes'],
        activation='relu',
        solver='adam',
        alpha=p['alpha'],
        batch_size=p['batch_size'],
        learning_rate='adaptive',
        learning_rate_init=p['learning_rate_init'],
        max_iter=p['max_iter'],
        early_stopping=True,
        validation_fraction=0.05,
        n_iter_no_change=5,
        random_state=RANDOM_STATE,
        verbose=False,
    )

    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ('x_scaler', StandardScaler(with_mean=True, with_std=True)),
            ('mlp', base_mlp),
        ]),
        transformer=StandardScaler(with_mean=True, with_std=True)
    )

    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_mae = float(mean_absolute_error(y_val, val_pred))
    val_rmse = float(np.sqrt(mean_squared_error(y_val, val_pred)))
    elapsed = float(time.perf_counter() - t0)

    row = {
        'trial': idx,
        **{k: str(v) for k,v in p.items()},
        'val_mae': val_mae,
        'val_rmse': val_rmse,
        'train_seconds': elapsed,
    }
    trial_rows.append(row)

    if best is None or val_mae < best['val_mae']:
        best = {'params': p, 'val_mae': val_mae, 'val_rmse': val_rmse}

    print(f"[{idx}/{len(trials)}] val_mae={val_mae:.4f} val_rmse={val_rmse:.4f} time={elapsed:.1f}s params={p}")

trials_df = pd.DataFrame(trial_rows).sort_values('val_mae')
print('Best by val MAE:')
print(trials_df.head(3))

tuning_seconds = float(time.perf_counter() - tuning_t0)
print(f'Tuning loop time: {tuning_seconds:.2f} sec ({tuning_seconds/60:.2f} min)')


## Step 6: Final Refit and Test Evaluation

Retrains with best hyperparameters on train+val, then evaluates on holdout test.


In [ ]:
final_stage_t0 = time.perf_counter()
# Refit best on train+val, evaluate on test
X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)

bp = best['params']
base_mlp = MLPRegressor(
    hidden_layer_sizes=bp['hidden_layer_sizes'],
    activation='relu',
    solver='adam',
    alpha=bp['alpha'],
    batch_size=bp['batch_size'],
    learning_rate='adaptive',
    learning_rate_init=bp['learning_rate_init'],
    max_iter=bp['max_iter'],
    early_stopping=True,
    validation_fraction=0.05,
    n_iter_no_change=5,
    random_state=RANDOM_STATE,
    verbose=False,
)

best_model = TransformedTargetRegressor(
    regressor=Pipeline([
        ('x_scaler', StandardScaler(with_mean=True, with_std=True)),
        ('mlp', base_mlp),
    ]),
    transformer=StandardScaler(with_mean=True, with_std=True)
)

t0 = time.perf_counter()
best_model.fit(X_train_full, y_train_full)
final_train_seconds = time.perf_counter() - t0

test_pred = best_model.predict(X_test)
mae = float(mean_absolute_error(y_test, test_pred))
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))

print(f'Test RMSE: {rmse:.4f}')
print(f'Test MAE : {mae:.4f}')
print(f'Final fit time: {final_train_seconds:.2f} sec ({final_train_seconds/60:.2f} min)')

final_stage_seconds = float(time.perf_counter() - final_stage_t0)
print(f'Final stage total time: {final_stage_seconds:.2f} sec ({final_stage_seconds/60:.2f} min)')


## Step 7: Save Artifacts

Writes trial table, model, and metrics JSON for reporting/reproducibility.


In [ ]:
# Save artifacts
import joblib

trials_path = ARTIFACT_DIR / 'trials.csv'
trials_df.to_csv(trials_path, index=False)

model_path = ARTIFACT_DIR / 'mlp_model_tuned.joblib'
joblib.dump(best_model, model_path)

total_notebook_seconds = float(time.perf_counter() - NOTEBOOK_T0)

metrics = {
    'data_path': str(DATA_PATH),
    'model': 'MLPRegressor_tuned',
    'holdout_days': HOLDOUT_DAYS,
    'val_days': VAL_DAYS,
    'n_trials': N_TRIALS,
    'best_params': {k: (list(v) if isinstance(v, tuple) else v) for k,v in bp.items()},
    'best_val_mae': float(best['val_mae']),
    'best_val_rmse': float(best['val_rmse']),
    'test_mae': mae,
    'test_rmse': rmse,
    'final_train_seconds': float(final_train_seconds),
    'timing': {
        'data_prep_seconds': float(data_prep_seconds),
        'tuning_seconds': float(tuning_seconds),
        'final_stage_seconds': float(final_stage_seconds),
        'total_notebook_seconds': float(total_notebook_seconds),
    },
}
metrics_path = ARTIFACT_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved:')
print('-', trials_path)
print('-', model_path)
print('-', metrics_path)

print(f"Total notebook runtime: {total_notebook_seconds:.2f} sec ({total_notebook_seconds/60:.2f} min)")
